# TRÍCH XUẤT VECTOR ĐẶC TRƯNG & TẠO DATABASE
**Mô tả:** Notebook này dùng để đọc ảnh sinh viên từ thư mục `Dataset/`, trích xuất vector đặc trưng 512 chiều.

**Mô hình:** Sử dụng YOLOv8-face để nhận dạng (detect) và InsightFace để trích xuất vector 512 chiều, sau đó lưu vào **SQLite Database**.

---

### 1. Khai báo thư viện và Khởi tạo Mô hình AI
* **YOLOv8-face:** Dùng để phát hiện vùng khuôn mặt (Bounding Box).
* **InsightFace:** Dùng để trích xuất vector đặc trưng (ArcFace).
* **Database:** Đồng bộ vào bảng `students` trong SQLite.


In [ ]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
import insightface
from insightface.app import FaceAnalysis
from database import init_db, save_student

# 1. Khởi tạo Database SQLite
init_db()

# 2. Khởi tạo mô hình YOLOv8-face (Detection)
yolo_model = YOLO('yolov8n-face.pt')

# 3. Khởi tạo mô hình InsightFace (Extraction 512D)
app = FaceAnalysis(name='buffalo_sc', providers=['CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

print("Đã tải xong model YOLOv8 và InsightFace, và kết nối Database!")


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/trietnguyen/.insightface/models/buffalo_sc/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/trietnguyen/.insightface/models/buffalo_sc/w600k_mbf.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
✅ Đã tải xong model YOLOv8 và InsightFace, và kết nối Database!


---
### 2. Trích xuất Vector và Đồng bộ Database (`database.db`)
* **Quy trình:**
  1. Duyệt qua từng thư mục sinh viên trong `Dataset/`.
  2. Dùng YOLOv8 để tìm khuôn mặt, cắt (crop) ảnh mặt.
  3. Dùng InsightFace trên ảnh đã cắt để lấy vector đặc trưng 512D (`normed_embedding`).
  4. Tính vector trung bình cho mỗi sinh viên để tăng độ chính xác.
  5. Đồng bộ vào SQLite thay vì file `.pkl`.


In [ ]:
dataset_dir = "Dataset" if os.path.exists("Dataset") else "dataset"

# Duyệt qua từng thư mục sinh viên trong dataset
for student_folder in os.listdir(dataset_dir):
    folder_path = os.path.join(dataset_dir, student_folder)
    
    if not os.path.isdir(folder_path):
        continue
        
    print(f"🔄 Đang xử lý dữ liệu cho: {student_folder}...")
    vectors = []
    
    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            continue
            
        # Nhận diện khuôn mặt bằng YOLOv8
        results = yolo_model(img, verbose=False)
        
        for r in results:
            boxes = r.boxes
            for box in boxes:
                # Cắt khuôn mặt theo Bounding Box của YOLOv8
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                h, w, _ = img.shape
                # Mở rộng bbox một chút để insightface dễ bắt landmarks
                padding = 10
                x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
                x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
                
                face_img = img[y1:y2, x1:x2]
                
                if face_img.size == 0:
                    continue
                
                # Truyền mặt đã cắt qua InsightFace để trích xuất vector 512 chiều
                faces = app.get(face_img)
                
                if len(faces) > 0:
                    embedding = faces[0].normed_embedding
                    vectors.append(embedding)
                else:
                    # Fallback nếu crop quá sát
                    faces_full = app.get(img)
                    if len(faces_full) > 0:
                        vectors.append(faces_full[0].normed_embedding)
            
    # Tính vector trung bình và lưu vào Database
    if len(vectors) > 0:
        mean_vector = np.mean(vectors, axis=0)
        mean_vector = mean_vector / np.linalg.norm(mean_vector) # Chuẩn hóa
        
        # Lưu vào SQLite thông qua file database.py
        save_student(student_folder, student_folder, mean_vector)
        print(f"   └─Đã trích xuất xong cho {student_folder} ({len(vectors)} ảnh hợp lệ)")
    else:
        print(f"   Không tìm thấy mặt hợp lệ cho {student_folder}")

print("\nDữ liệu đã được đồng bộ vào SQLite Database thành công.")


🔄 Đang xử lý dữ liệu cho: HoangNMSE200102...
   └─ ✅ Đã trích xuất xong cho HoangNMSE200102 (2 ảnh hợp lệ)

Dữ liệu đã được đồng bộ vào SQLite Database thành công.


/Users/trietnguyen/Downloads/project-MAI391/.venv/lib/python3.12/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
